# Systematic Trading Strategies with ML — Meta-Model Submission
## Part I — Foundations: EDA, external data & feature engineering

**Imperial College London × Alken Asset Management.** This notebook builds a **meta-model**. It takes
the provided primary trading signal `s ∈ {-1, 0, +1}` for 11 futures and, for each non-zero signal,
predicts the **probability that following the trade is profitable** under a triple-barrier exit.

### How to run this notebook
1. To run on data extended through Dec 2022, replace `data/ohlcv_data.csv` and `data/primary_signals.csv`
   with the extended versions. The notebook detects the longer history on its own and regenerates every
   engineered feature for the hidden Jul–Dec 2022 window.
2. Install dependencies. **With uv:** `uv sync --group features-extra`. **Without uv:**
   `pip install -r requirements.txt && pip install -e .` (see `README.md`).
3. Run top to bottom (*Kernel → Restart & Run All*). Part I rebuilds the feature matrix **live** in
   Section 0 and writes it to `results/feature_matrix.parquet`. Part II then loads the team's committed
   CPCV results and saved hyper-parameters, so it reproduces the meta-model numbers in seconds.

> **Scope — Part I (feature engineering).** EDA of the raw prices and the signal, the external macro
> dataset (F11), the learned HMM regime features (F3 and F17), and the full F1–F17 feature catalogue.
> Every feature is **regenerable from raw inputs**, uses only information available at the time, and is
> fitted on the FE-train block before the boundary.

> **Scope — Part II (meta-model).** Section 6 builds the triple-barrier labels, with the barrier
> geometry chosen by grid search. Section 7 compares four model families, both **pooled and
> per-instrument**, under CPCV, and locks one champion and feature variant per instrument. Section 8
> saves that selection. Section 9 extracts the cluster-level feature importance of each champion.
> Section 10 evaluates the calibrated model out of sample. Section 11 turns the calibrated probabilities
> into a sized strategy and compares it against the primary signal. Two items are left for a later step:
> the consolidated `date,instrument,prediction` deliverable and the refit that predicts the hidden
> H2-2022 window.

## Section 0 — Setup & data load

In [ ]:
%matplotlib inline
import json, re, collections, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
np.random.seed(42); SEED = 42
plt.rcParams["figure.dpi"] = 100
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 40)

from stml.io import _find_repo_root, load_clean_data, load_returns_panel
from stml import na_checks as nc
from stml.model.dataset import close_panel, events_frame
from stml.metamodel.scope import ASSET_CLASS_MAP

ROOT = _find_repo_root(Path.cwd().resolve())
DATA, RESULTS, REPORTS = ROOT / "data", ROOT / "results", ROOT / "reports"
RESULTS.mkdir(exist_ok=True)

ohlcv, signals = load_clean_data()      # ohlcv: long OHLCV ; signals: wide (date + 11 instrument cols)
close_w = close_panel()                 # wide close prices, date-indexed, instrument columns
ret_w   = load_returns_panel()          # wide daily LOG returns, full price history
sig_w   = signals.set_index("date")     # wide signals, date-indexed
INSTR   = list(nc.INSTRUMENTS)          # canonical 11-instrument order
CLASS   = dict(ASSET_CLASS_MAP)         # {instrument -> 'EQ'/'EN'/'ME'} (from stml.metamodel.scope)

# Feature-engineering partition boundaries, pinned by DATE so they never move when the signal axis is
# extended with the hidden Jul-Dec 2022 block (rows after TEST_END are tagged "oos").
FE_TRAIN_END = pd.Timestamp("2021-07-01")
VAL_END      = pd.Timestamp("2021-12-30")
TEST_END     = pd.Timestamp("2022-06-30")

TICKER_NAMES = {
    "cl1s": "Crude Oil (CL)",   "es1s": "S&P 500 e-mini (ES)", "fesx1s": "Euro Stoxx 50 (FESX)",
    "gc1s": "Gold (GC)",        "hg1s": "Copper (HG)",         "ho1s": "Heating Oil (HO)",
    "ng1s": "Natural Gas (NG)", "nq1s": "Nasdaq 100 (NQ)",     "pl1s": "Platinum (PL)",
    "rb1s": "RBOB Gasoline (RB)", "si1s": "Silver (SI)",
}

sig_max = sig_w.index.max()
IS_EXTENDED = sig_max > TEST_END
print(f"loaded {len(ohlcv):,} OHLCV rows | signal window {sig_w.index.min().date()} -> {sig_max.date()}")
print(f"FE-train boundary <= {FE_TRAIN_END.date()} (frozen) | seed {SEED}")
print("hidden OOS block present (signals extend past 2022-06-30):", bool(IS_EXTENDED))

In [ ]:
# The notebook REBUILDS the full feature matrix from raw OHLCV+signals, which needs the two optional
# "features-extra" dependencies: hmmlearn (F17 regimes) and PyWavelets (F13 multiscale energy).
_missing = []
for _mod, _pkg in [("hmmlearn", "hmmlearn"), ("pywt", "pywavelets")]:
    try:
        __import__(_mod)
    except Exception:
        _missing.append(_pkg)
if _missing:
    raise RuntimeError(
        "Missing required feature dependencies: " + ", ".join(_missing)
        + "\nInstall with:  uv sync --group features-extra"
        + "   (or)   pip install " + " ".join(_missing))
print("feature dependencies present: hmmlearn, pywavelets")

### 0.1 — Regenerate the feature matrix (live, extended-axis aware)

The matrix is rebuilt **from the raw OHLCV and signals in `data/`**. So when the extended files (through
Dec 2022) replace the originals, running the notebook regenerates every engineered feature for the
hidden Jul–Dec 2022 window here.

`FeaturePipeline.fit()` fits the **TF** families (F3 and F17 per instrument, F4 per asset class, and the
F11 macro family) on the **FE-train partition only (≤ 2021-07-01)** and then freezes them. The boundary
is pinned by *date*, so extending the data never moves it. `.transform()` then applies the engineered
**E** families and the frozen **TF** transforms causally. It tags each row's `partition` (`train`,
`val`, `test`, and **`oos`** for any row after 2022-06-30) and keeps only the nonzero-signal trade-days.

**F11 is the only family that needs external (macro) data.** Its hidden-window values are supplied
pre-computed in `data/features/f11_macro_context_oos.csv`, z-scored with the same FE-train statistics as
the rest, and spliced onto the `oos` rows below. The macro source workbook is therefore not needed for
the hidden period.

In [ ]:
from stml.metamodel.pipeline import FeaturePipeline
from stml.metamodel.catalog import assert_coverage

# ~1-3 minutes: fits all TF families on FE-train and transforms causally over the full (possibly
# extended) axis. macro_path supplies the F11 workbook for the released window.
pipe   = FeaturePipeline(macro_path=str(DATA / "additional_data.xlsx")).fit(ohlcv, signals)
matrix = pipe.transform(ohlcv, signals)
assert_coverage(matrix.columns)                       # every catalogued feature column is present
assert matrix.attrs.get("fe_train_end_date") == "2021-07-01"

part_counts = matrix["partition"].value_counts().to_dict()
print(f"feature matrix: {matrix.shape[0]:,} nonzero-signal rows x {matrix.shape[1]} cols "
      f"({matrix.shape[1] - 4} features + 4 metadata)")
print("partition row counts:", part_counts)

In [ ]:
# Splice the shipped external macro features (F11) onto the hidden-window (oos) rows.
from stml.metamodel.macro_features import macro_feature_columns
f11_cols = macro_feature_columns()
n_oos = int((matrix["partition"] == "oos").sum())
if n_oos:
    ext = (pd.read_csv(DATA / "features" / "f11_macro_context_oos.csv", parse_dates=["date"])
             .set_index(["date", "instrument"])[f11_cols])
    m = matrix.set_index(["date", "instrument"])
    common = m.index[m["partition"] == "oos"].intersection(ext.index)
    m.loc[common, f11_cols] = ext.loc[common, f11_cols]
    matrix = m.reset_index()
    note = "" if len(common) == n_oos else "  (unmatched oos rows keep median-imputable NaNs)"
    print(f"F11 OOS splice: filled {len(common):,}/{n_oos:,} oos rows from the shipped external CSV{note}")
else:
    print("No oos rows (released-window run) -> F11 splice is a no-op.")

In [ ]:
# Persist the regenerated matrix + a live provenance summary (computed, not read from a committed file).
out_path = RESULTS / "feature_matrix.parquet"
matrix.to_parquet(out_path)
prov = {
    "fe_train_end_date": pipe.fe_train_end,
    "seed": pipe.seed,
    "n_rows": int(len(matrix)),
    "n_feature_cols": int(matrix.shape[1] - 4),
    "partition_row_counts": {k: int(v) for k, v in matrix["partition"].value_counts().items()},
}
print(f"wrote {out_path.relative_to(ROOT)}  ({prov['n_rows']:,} rows x {prov['n_feature_cols']} features)")
print("provenance:", json.dumps(prov))

## Section 1 — Exploratory data analysis

### 1.1 — Coverage: a two-window dataset

The 11 instruments span three asset classes: **Equity** (`es1s`, `nq1s`, `fesx1s`), **Energy** (`cl1s`,
`ho1s`, `rb1s`, `ng1s`) and **Metals** (`gc1s`, `si1s`, `hg1s`, `pl1s`). The commodity contracts carry
about 30 years of price history, while the equity indices start later. The **primary signal**, however,
exists only for **2020-01 to 2022-06**. The decades of earlier prices are there only to warm up trailing
features causally; all modelling happens inside the short signal window.

In [ ]:
# Data-availability heatmap (one row per instrument; red lines mark the signal window).
avail = close_w.notna().astype(int).T          # instrument x date
fig, ax = plt.subplots(figsize=(13, 3.2))
ax.imshow(avail.values, aspect="auto", cmap="Greys", interpolation="nearest")
ax.set_yticks(range(len(avail.index))); ax.set_yticklabels(avail.index)
yrs = pd.to_datetime(avail.columns).year
yr_changes = np.where(np.diff(yrs) != 0)[0] + 1
step = max(1, len(yr_changes) // 12)
ax.set_xticks(yr_changes[::step]); ax.set_xticklabels([str(yrs[i]) for i in yr_changes[::step]], rotation=45)
s0 = avail.columns.get_indexer([signals.date.min()], method="nearest")[0]
s1 = avail.columns.get_indexer([signals.date.max()], method="nearest")[0]
ax.axvline(s0, color="C3", lw=1.2, label="signal window"); ax.axvline(s1, color="C3", lw=1.2)
ax.set_title("OHLCV data availability per instrument  (red = signal window)")
ax.legend(loc="lower left"); plt.tight_layout(); plt.show()

n_eq = sum(CLASS[i] == "EQ" for i in INSTR)
print(f"{len(INSTR)} instruments: {len(INSTR)-n_eq} commodities (~30y history), {n_eq} equity indices; "
      f"signal window {sig_w.index.min().date()} -> {sig_w.index.max().date()}")

### 1.2 — Return structure: stationary but heavy-tailed

Daily log returns are **stationary** (every instrument rejects the ADF unit-root null) but clearly
**non-Gaussian**: excess kurtosis is well above 0 everywhere, which means fat tails. This is why we lean
on robust and tree-based models later, and why the triple-barrier exits used for labelling are **scaled
by volatility**.

In [ ]:
# Per-instrument return summary over the full price history (log returns).
ANN = np.sqrt(252)
ret_summary = pd.DataFrame({
    "name":          pd.Series(TICKER_NAMES),
    "mean_daily_bp": ret_w.mean() * 1e4,
    "ann_vol_%":     ret_w.std() * ANN * 100,
    "skew":          ret_w.skew(),
    "kurtosis":      ret_w.kurt(),          # excess kurtosis; >> 0 = fat tails
    "min_day_%":     ret_w.min() * 100,
    "max_day_%":     ret_w.max() * 100,
    "n_obs":         ret_w.count(),
}).reindex(INSTR).round(2)
display(ret_summary)

In [ ]:
# Log-return distributions with a Gaussian overlay (log y-axis exposes the tails).
fig, axes = plt.subplots(3, 4, figsize=(14, 8))
for ax, inst in zip(axes.flat, INSTR):
    r = ret_w[inst].dropna()
    ax.hist(r, bins=80, density=True, alpha=0.7, color="C0")
    xs = np.linspace(r.quantile(0.001), r.quantile(0.999), 200)
    ax.plot(xs, (1/(r.std()*np.sqrt(2*np.pi))) * np.exp(-0.5*((xs-r.mean())/r.std())**2),
            color="C3", lw=1.2, label="N(mu,sig^2)")
    ax.set_title(f"{inst}  kurt={r.kurt():.1f}", fontsize=10); ax.set_yscale("log")
    ax.legend(fontsize=7)
for ax in axes.flat[len(INSTR):]:
    ax.set_visible(False)
fig.suptitle("Log-return distributions (log y, Gaussian overlay) — fat tails everywhere", y=1.02)
plt.tight_layout(); plt.show()

### 1.3 — Cross-sectional structure: three blocks

Clustering the full-sample log-return correlation matrix reveals **three blocks**: energy, metals and
equity. Correlation within a block is about 0.7–0.9, while correlation across blocks is weak and
regime-dependent. This panel structure has two consequences. Cross-validation must be **blocked**,
because a fold that leaks one block also leaks its correlated neighbours. And the cross-sectional **F9**
features are computed over the whole universe.

In [ ]:
import seaborn as sns
# corr_max_info: pairwise-complete (max data, not truncated to shortest history) + PSD-repaired.
corr = nc.corr_max_info(ret_w, min_periods=252)
g = sns.clustermap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1,
                   figsize=(8, 8), cbar_pos=(0.02, 0.85, 0.03, 0.12))
g.fig.suptitle("Daily log-return correlation (pairwise-complete, PSD-repaired, clustered)", y=1.02)
plt.show()

### 1.4 — The primary signal: imbalanced, persistent, near-independent

The signal is strongly **imbalanced and instrument-specific**: `ho1s` is flat on about 90% of days,
`ng1s` is **never long**, and `es1s` is long about 70% of the time. It is also strongly **persistent**,
with lag-1 autocorrelation of 0.6–0.9 and a mean run length above 5 days. Because of this persistence
the rows are **not** independent, so **purged and blocked cross-validation is required**. Correlation of
the signal across instruments is low, so each instrument carries close to independent information.

In [ ]:
# Class balance + persistence per instrument.
long_sig = signals.melt(id_vars="date", var_name="instrument", value_name="s")
bal = (long_sig.groupby("instrument")["s"].value_counts(normalize=True)
       .unstack(fill_value=0).round(3))
bal.columns = [f"p(s={int(c)})" for c in bal.columns]
rows = []
for inst in INSTR:
    s = sig_w[inst]
    runs = (s != s.shift()).cumsum()
    rl = s[s != 0].groupby(runs).size()
    rows.append({"instrument": inst, "lag1_autocorr": round(s.autocorr(1), 3),
                 "mean_run_len": round(rl.mean(), 1) if len(rl) else 0.0,
                 "nonzero_frac": round((s != 0).mean(), 3)})
persist = pd.DataFrame(rows).set_index("instrument")
display(bal.join(persist).reindex(INSTR))

# Stacked class-share bar.
shares = pd.DataFrame({
    "short (-1)": (sig_w == -1).mean(), "flat (0)": (sig_w == 0).mean(), "long (+1)": (sig_w == 1).mean(),
}).reindex(INSTR)
ax = shares.plot(kind="bar", stacked=True, figsize=(11, 3.6),
                 color=["#d62728", "#bdbdbd", "#2ca02c"])
ax.set_ylabel("share of days"); ax.set_ylim(0, 1)
ax.set_title("Signal class balance per instrument (2020–2022)")
ax.legend(loc="center left", bbox_to_anchor=(1.0, 0.5)); plt.tight_layout(); plt.show()

### 1.5 — Signal × forward returns: the signal predicts the *next* bar

This is the most important EDA finding for labelling, and two views agree on it:

- **Lead/lag.** `corr(s_t, r_{t+k})` is strongest at **k = +1**, so the signal *leads* the return.
- **Signed-return Sharpe by lag.** The quantity `g_i(L) = s_i · u_{t+L}` (from the labelling study) is
  *negative* at **lag 0** (same day) for most instruments, **strongest at lag 1**, and decays by lag 2.
  This is a placebo-in-time pattern: the edge appears only at the tradeable lag.

Put together, the signal is a **short-horizon mean-reversion / counter-trend** call that pays off on the
**next** bar. The consequence for labelling is concrete: **enter at `t+1`, not `t`** (entering at `t`
would use information from the future). Three thin or awkward names to watch are **`cl1s`** (the
strongest, with label-1 ≈ 0.70), **`ng1s`** (short-only) and **`ho1s`** (about 60 events, so
statistically weak).

In [ ]:
# (a) Per-instrument corr(s_t, r_{t+1}), mean pnl and hit rate.
rets_s = close_w.pct_change()
hit = []
for inst in INSTR:
    s = sig_w[inst].reindex(close_w.index); fwd = rets_s[inst].shift(-1)
    m = (s != 0) & fwd.notna(); pnl = (s * fwd)[m]
    hit.append({"instrument": inst, "corr_s_rfwd": round(float(s[m].corr(fwd[m])), 3),
                "mean_pnl_bp": round(float(pnl.mean()) * 1e4, 2),
                "hit_rate": round(float((pnl > 0).mean()), 3), "n": int(m.sum())})
display(pd.DataFrame(hit).set_index("instrument").reindex(INSTR))
print("corr(s_t, r_{t+1}) is mostly positive -> the signal predicts the NEXT bar (counter-trend flavour).")

In [ ]:
# (b) Lead/lag: corr(signal_t, r_{t+k}) for k in [-5..5].  k>0 = signal leads return = predictive.
lags = range(-5, 6)
lead_lag = pd.DataFrame(index=lags, columns=INSTR, dtype=float)
for k in lags:
    aligned = ret_w.shift(-k).reindex(sig_w.index)
    for inst in INSTR:
        a = sig_w[inst].astype(float); b = aligned[inst]
        ok = a.notna() & b.notna()
        if ok.sum() > 30 and a[ok].std() > 0 and b[ok].std() > 0:
            lead_lag.loc[k, inst] = a[ok].corr(b[ok])
fig, ax = plt.subplots(figsize=(11, 5))
for inst in INSTR:
    ax.plot(lead_lag.index, lead_lag[inst], marker="o", lw=1.0, label=inst)
ax.axvline(0, color="k", lw=0.5); ax.axhline(0, color="k", lw=0.5)
ax.set_xlabel("lag k:  k>0 => signal leads return (predictive)")
ax.set_ylabel("corr(signal_t, r_{t+k})")
ax.set_title("Lead/lag: is the signal predictive (k>0) or reactive (k<0)?")
ax.legend(ncol=4, fontsize=8); plt.tight_layout(); plt.show()

In [ ]:
# (c) Raw signed-return Sharpe by lag (no geometry / no label filter) — the placebo-in-time check.
from stml.model.evaluate import nav_sharpe
ev = events_frame(matrix)        # [date, instrument, side, sigma] over the released window

def lagged_signed_returns(events, close_wide, lags=(0, 1, 2)):
    out = events[["date", "instrument", "side"]].copy().reset_index(drop=True)
    for L in lags: out[f"g{L}"] = np.nan
    for inst, grp in out.groupby("instrument"):
        if inst not in close_wide.columns: continue
        s = close_wide[inst].dropna(); u = s.pct_change().to_numpy()
        pos = s.index.get_indexer(pd.DatetimeIndex(grp["date"])); side = grp["side"].to_numpy(float)
        for L in lags:
            vals = np.full(len(grp), np.nan)
            ok = (pos >= 0) & (pos + L >= 0) & (pos + L < len(u))
            vals[ok] = u[pos[ok] + L] * side[ok]
            out.loc[grp.index, f"g{L}"] = vals
    return out

def raw_lag_table(G):
    rows = []
    for inst, grp in G.groupby("instrument"):
        for L in (0, 1, 2):
            r = grp[f"g{L}"].to_numpy(float); r = r[np.isfinite(r)]
            sh = nav_sharpe(pd.DataFrame({"ret": r}), np.ones(len(r), bool))["sharpe"]
            rows.append({"instrument": inst, "lag": L, "sharpe": round(sh, 3)})
    return pd.DataFrame(rows).pivot(index="instrument", columns="lag", values="sharpe")

G = lagged_signed_returns(ev, close_w)
print("RAW primary Sharpe by lag (no geometry, no filter) — expect lag 1 strongest (placebo-in-time):")
display(raw_lag_table(G).reindex(INSTR))

### 1.6 — EDA takeaways for the modelling that follows

- **Two windows.** About 30 years of prices warm up the features, but the model itself lives in the
  2020-01 to 2022-06 signal window.
- **Per-instrument imbalance.** Calibrate and threshold per instrument, and watch for single-class
  collapse.
- **Persistence (autocorrelation 0.6–0.9).** Use **purged and embargoed** blocked CV, never random
  k-fold.
- **Panel structure (three correlated blocks).** Treat the data as a panel and block the CV across the
  whole cross-section.
- **Enter at `t+1`.** The signal predicts the next bar, and the labelling entry must respect that.
- **Fat tails.** Prefer robust and tree-based models, and volatility-scaled barriers.

## Section 2 — External macro dataset (feature family F11)

**F11 is the only feature family built from external data.** We sourced a cross-asset macro workbook
(`data/additional_data.xlsx`, with Bloomberg-style daily, weekly and monthly series) to give the
meta-model *context* that the price-only families cannot see, such as whether a counter-trend trade is
firing into a calm tape or a credit-stress spike.

**Our hypothesis is intentionally simple.** Rather than data-mine the whole workbook, we curated a
compact set that covers the major cross-asset risk dimensions, chosen so a model can weight them:

1. equity volatility
2. interest rates
3. USD strength
4. credit spreads
5. inflation expectations
6. energy inventories
7. manufacturing demand

In [ ]:
from stml.metamodel.macro_features import (
    load_macro_raw, KEEP, SPREAD_INPUTS, SPREADS, DROPPED, MOMENTUM,
    compute_availability, macro_feature_columns)

# All named series physically present in the workbook (paired date+value columns).
_hdr = pd.read_excel(str(DATA / "additional_data.xlsx"), nrows=1)
all_series = sorted(c for c in _hdr.columns
                    if not str(c).lower().startswith("unnamed") and "date" not in str(c).lower())
# load_macro_raw curates these down to the series F11 actually uses (standalone + spread inputs);
# the dropped series are not loaded.
raw = load_macro_raw(str(DATA / "additional_data.xlsx"))
inv = pd.DataFrame([{"series": k, "n_obs": int(v.notna().sum()),
                     "start": v.dropna().index.min().date(), "end": v.dropna().index.max().date()}
                    for k, v in raw.items()]).sort_values("series").reset_index(drop=True)
print(f"workbook contains {len(all_series)} named macro series; F11 uses {len(raw)} of them "
      f"({len(KEEP)} standalone + {len(SPREAD_INPUTS)} spread inputs), drops {len(DROPPED)}, "
      f"and emits {len(macro_feature_columns())} feature columns")
print("all 22 series:", ", ".join(all_series))
print("\nspans of the series F11 loads:")
display(inv)

### 2.1 — Curation: from 22 raw series to 45 columns

From the workbook's 22 named series we **kept 12 as standalone series**, used **2 only as inputs to
spreads**, built **3 economically motivated spreads**, and **dropped 8** (either redundant with a series
we kept, or out of scope for this universe). That leaves 15 effective series. Each one yields **3
columns** — a point-in-time *level* plus two *momentum* (change) horizons — for a total of **15 × 3 =
45** F11 columns.

In [ ]:
rows = []
for s, (rcls, desc) in KEEP.items():
    rows.append({"series": s, "role": "standalone", "release": rcls, "captures": desc})
for s in SPREAD_INPUTS:
    rows.append({"series": s, "role": "spread-input only", "release": "", "captures": "used only inside a spread"})
for name, (a, b, rcls, desc) in SPREADS.items():
    rows.append({"series": f"{name} = {a} - {b}", "role": "computed spread", "release": rcls, "captures": desc})
for s in DROPPED:
    rows.append({"series": s, "role": "dropped", "release": "", "captures": "redundant / out-of-scope for this universe"})
curation = pd.DataFrame(rows)
print(f"22 raw series -> {len(KEEP)} standalone + {len(SPREADS)} spreads = {len(KEEP)+len(SPREADS)} effective "
      f"x 3 cols = {len(macro_feature_columns())} F11 columns "
      f"({len(SPREAD_INPUTS)} spread-input-only, {len(DROPPED)} dropped)")
display(curation)

### 2.2 — Point-in-time availability (no look-ahead)

A macro value can enter a trade-day row only **once it has actually been published**. Each series has a
release cadence, and that cadence sets both its availability lag and its momentum horizons:

| class | availability rule | momentum (business days) |
|---|---|---|
| `daily` (market series) | same-day EOD close (lag 0) | 5, 20 |
| `weekly_eia` (inventories) | Friday week-ending stamp **+ 6 calendar days** (a conservative buffer past the ~Wed/Thu release) | 5, 20 |
| `monthly_pmi` (PMIs) | month-end stamp **+ 1 business day** (the release) | 21, 63 |

The z-score standardiser is **fit on the FE-train slice only (≤ 2021-07-01) and then frozen** going
forward. The 45 columns are **broadcast identically to all 11 instruments**, because they describe
global macro context. Under the leakage taxonomy in Section 4, F11 is therefore a **fitted (TF)**
family.

In [ ]:
# Surface the availability rule straight from the code (one representative stamp per class).
demos = [("daily", pd.Timestamp("2021-08-06")),
         ("weekly_eia", pd.Timestamp("2021-08-06")),   # a Friday week-ending stamp
         ("monthly_pmi", pd.Timestamp("2021-07-31"))]  # a month-end stamp
lag_rows = []
for rcls, stamp in demos:
    avail = compute_availability(stamp, rcls)
    lag_rows.append({"release_class": rcls, "stamp": stamp.date(), "available_on": avail.date(),
                     "lag_days": (avail - stamp).days, "momentum_bdays": MOMENTUM[rcls]})
display(pd.DataFrame(lag_rows))

In [ ]:
# The realised F11 columns in the feature matrix, and proof of the global broadcast.
f11 = [c for c in matrix.columns if c.startswith("f11_")]
by_suffix = collections.Counter(c.rsplit("_", 1)[1] for c in f11)
print(f"{len(f11)} f11_* columns; by suffix: {dict(by_suffix)}")

d0 = matrix["date"].iloc[len(matrix) // 2]
two = (matrix[(matrix.date == d0) & (matrix.instrument.isin(["cl1s", "es1s"]))]
       .set_index("instrument")[f11[:6]].T)
print(f"Same date ({pd.Timestamp(d0).date()}), two different instruments -> identical macro values:")
display(two.round(3))

vix = matrix[matrix.instrument == "cl1s"].set_index("date")["f11_vix_level"].sort_index()
ax = vix.plot(figsize=(11, 2.6), color="C3")
ax.axhline(0, color="k", lw=0.5)
ax.set_title("f11_vix_level (FE-train z-scored) across the released window")
plt.tight_layout(); plt.show()

**How F11 feeds the model.** These 45 standardised, publication-lagged columns are joined onto *every*
instrument-date row. A model can then learn macro-conditional adjustments to the per-instrument signal —
for example, to trust a counter-trend trade less when MOVE and HY-OAS are both spiking.

## Section 3 — HMM regime features (families F3 & F17)

Two families summarise the **volatility regime** each row sits in:

- **F3** is a **2-regime** pair: a Gaussian Mixture model and a Markov-switching model. It reports the
  high-vol posterior, the switch intensity, and the dwell time.
- **F17** is a **3-state Gaussian HMM** (low, mid and high vol) whose **transition matrix** links the
  states across time.

Both are fit **per instrument** (11 separate models each), on the **FE-train partition only**
(≤ 2021-07-01), and then **frozen**. They are not pooled across the universe (unlike F11) and not fit per
asset class (unlike the F4 latent stack). Both emit strictly **causal, forward-filtered** posteriors
`P(state | data ≤ t)`. We never use `hmmlearn`'s smoothed `predict_proba`, which would look ahead to
`t+1 … T`. Both are **fitted (TF)** families.

**Observation vector (2-D, per instrument):** `ret` is the daily log return, and `vol` is the trailing
20-day **annualised** realised volatility. F17 consumes `(ret, vol)` directly. F3's GMM standardises
`(ret, vol)` with frozen FE-train statistics, while its Markov model runs on `ret` alone.

In [ ]:
# The FeaturePipeline fitted in Section 0 already holds every per-instrument HMM/regime bundle, frozen
# on the FE-train block -- we reuse it here (no second fit) to inspect the learned regimes.
REP_INST = "cl1s"   # representative: energy, full history, strongest signal (label-1 ~ 0.70)
print(f"Reusing the Section-0 FeaturePipeline. Showing {REP_INST} in depth, then an 11-instrument summary.")

In [ ]:
# F17 — the 3-state HMM for the representative instrument. ALWAYS reorder arrays by `bundle.order`
# (states sorted by ascending FE-train mean vol) so lo/mid/hi are comparable; raw EM order is arbitrary.
if pipe is not None:
    b = pipe._hmm[REP_INST]; order = b.order; names = ["lo", "mid", "hi"]
    T     = pd.DataFrame(b.hmm.transmat_[np.ix_(order, order)], index=names, columns=names)
    means = pd.DataFrame(b.hmm.means_[order], index=names, columns=["ret", "vol"])
    start = pd.Series(b.hmm.startprob_[order], index=names, name="startprob")
    covs  = np.asarray(b.hmm.covars_)[order]
    disp  = pd.DataFrame({"sd_ret": np.sqrt(covs[:, 0, 0]), "sd_vol": np.sqrt(covs[:, 1, 1])}, index=names)
    print(f"F17 — {REP_INST}: 3-state Gaussian HMM fit on {len(b.train_index)} FE-train days")
    print("\nTransition matrix  P(to | from)   (rows = from-state, cols = to-state):")
    display(T.round(3))
    print("Per-regime emission means  (raw log return, annualised vol) — confirms lo < mid < hi vol:")
    display(means.round(4))
    print("Start probabilities & per-regime dispersion:")
    display(pd.concat([start, disp], axis=1).round(4))
    print("diag(T) =", T.values.diagonal().round(3), "-> high persistence: regimes are sticky.")

In [ ]:
# F3 — the GMM + Markov-switching pair for the same instrument (the two should agree on 'high-vol').
if pipe is not None:
    rb = pipe._regime[REP_INST]
    print(f"F3 — {REP_INST}: high-vol GMM component = {rb.gmm_highvol_comp} | "
          f"high-vol Markov regime = {rb.markov_highvol_regime}")
    gmm_means = pd.DataFrame(rb.gmm.means_, columns=["z_ret", "z_vol"],
                             index=[f"comp{i}" for i in range(rb.gmm.n_components)])
    print("GMM component means (in FROZEN-standardised z-space, not raw units):")
    display(gmm_means.round(3))
    print("Markov-switching variances (last 2 fitted params; the larger one is the high-vol regime):",
          np.round(np.asarray(rb.markov_params[-2:], float), 5))

In [ ]:
# 11-instrument summary: regime persistence (transition-matrix diagonal) + high-state vol per instrument.
if pipe is not None:
    rows = []
    for inst in INSTR:
        b = pipe._hmm[inst]
        rec = {"instrument": inst, "ok": b.ok, "n_train": len(b.train_index)}
        if b.ok:
            d = b.hmm.transmat_[np.ix_(b.order, b.order)].diagonal()
            rec.update({"persist_lo": round(float(d[0]), 3), "persist_mid": round(float(d[1]), 3),
                        "persist_hi": round(float(d[2]), 3),
                        "hi_state_mean_vol": round(float(b.hmm.means_[b.order][-1, 1]), 3)})
        rows.append(rec)
    display(pd.DataFrame(rows).set_index("instrument").reindex(INSTR))

### 3.1 — Derived columns and an example feature vector

Each family contributes **4 columns** to every nonzero-signal row:

- **F17:** `f17_hmm_state_lo`, `f17_hmm_state_mid`, `f17_hmm_state_hi` (the filtered posteriors, which
  form a simplex that sums to 1) and `f17_hmm_state_argmax` (the most-likely state, labelled 0/1/2).
- **F3:** `f3_gmm_prob_highvol`, `f3_markov_prob_highvol`, `f3_markov_switch_prob` (the trailing |Δ| of
  the high-vol probability, which measures switch intensity) and `f3_regime_dwell` (the number of days
  since the regime call last flipped).

These read straight out of the materialised matrix, with no fitting needed:

In [ ]:
hmm_cols = ["f3_gmm_prob_highvol", "f3_markov_prob_highvol", "f3_markov_switch_prob", "f3_regime_dwell",
            "f17_hmm_state_lo", "f17_hmm_state_mid", "f17_hmm_state_hi", "f17_hmm_state_argmax"]
row = matrix[matrix.instrument == REP_INST].dropna(subset=["f17_hmm_state_lo"]).iloc[100]
ex = row[["date", "instrument"] + hmm_cols].to_frame("value")
display(ex)
simplex = float(row[["f17_hmm_state_lo", "f17_hmm_state_mid", "f17_hmm_state_hi"]].sum())
print(f"F17 posteriors sum to {simplex:.6f} (simplex); argmax state = {int(row['f17_hmm_state_argmax'])} "
      f"(0=lo, 1=mid, 2=hi vol)")

**How the regime features feed the model.** These 8 causal, frozen columns are joined onto every row, so
a model can learn that the primary signal's reliability is **regime-dependent**. Counter-trend trades,
for instance, tend to work in calm regimes and break during high-vol transitions. Whether these columns
actually matter is tested at the cluster level in Part II §9.

## Section 4 — Feature engineering (families F1–F17)

The full feature layer is **175 features + 4 metadata = 179 columns** over **4,984 nonzero-signal
trade-days**, organised into **16 families** (`F1`–`F17`, with `F14` intentionally unused). It is a
de-duplicated union of three teammates' branches: the `F1`–`F11` base, plus folded-in newer families.
Those additions are `F12` and `F17` together with a Rogers–Satchell volatility estimator, and `F13`,
`F15` and `F16` together with several `F5`, `F7` and `F9` additions.

In [ ]:
from stml.metamodel.catalog import CATALOG, _FAMILY_TITLES
META = {"date", "instrument", "partition", "fe_train_end_date"}
feat = [c for c in matrix.columns if c not in META]

def family_of(col):
    name = col[2:] if col.startswith("z_") else col      # strip a z-twin prefix
    return re.match(r"(f\d+)_", name).group(1).upper()

recs = {}
for c in feat:
    f = family_of(c); lk = CATALOG[c].leakage_class
    r = recs.setdefault(f, {"n": 0, "E": 0, "TF": 0, "LI": 0})
    r["n"] += 1; r[lk] += 1
order = sorted(recs, key=lambda f: int(f[1:]))
fam_tbl = pd.DataFrame([{
    "family": f, "title": _FAMILY_TITLES[f].split(" — ", 1)[1], "n_cols": recs[f]["n"],
    "leakage": "/".join(f"{k}:{recs[f][k]}" for k in ("E", "TF", "LI") if recs[f][k]),
} for f in order]).set_index("family")
display(fam_tbl)

tot = collections.Counter(CATALOG[c].leakage_class for c in feat)
print(f"{len(feat)} features = E:{tot['E']} engineered + TF:{tot['TF']} fitted + LI:{tot['LI']} label-interface")

### 4.1 — Leakage taxonomy (the methodology backbone)

Every feature uses only information available at or before time `t`. We classify each column into one of
three leakage classes, and the test suite checks each one:

- **E — engineered.** No fitting. These are causal by **truncation-invariance**: the value at `t` is the
  same whether it is computed on `data[:t+1]` or on the full series.
- **TF — fitted.** The GMM, Markov, HMM, PCA, KMeans, autoencoder and scaler families (`F3`, `F4`,
  `F11`, `F16`, `F17`) are fit on the **FE-train partition only (≤ 2021-07-01)** and then applied
  causally with **frozen** parameters.
- **LI — label-interface.** The two columns the downstream triple-barrier label consumes: `f2_vol_20`
  (the barrier sigma) and `f5_trailing_run_length`.

In addition, every scale-dependent **E** column carries a parallel **`z_<col>` twin**: a per-instrument
*causal expanding-window* z-score that does not depend on the split. Bounded columns (ratios,
probabilities, sin/cos terms, percentiles) get no twin.

### 4.2 — The families, grouped (and tied back to the EDA)

- **Signal-aligned core (the highest-value group).** `F1` counter-trend / mean-reversion (led by
  `f1_mr_score_20`, the family the §1.5 lag-1 evidence predicts should dominate); `F5` signal-derived
  run structure (motivated by the persistence in §1.4); and `F6` momentum / trend-contrast.
- **Volatility and path.** `F2` volatility and dispersion (including the Rogers–Satchell estimator and
  the z-twins); `F12` Hurst, variance-ratio and efficiency-ratio path structure; `F13` wavelet
  multiscale energy; and `F15` conditional-risk / first-passage.
- **Regime and drift (fitted).** `F3` GMM+Markov and `F17` HMM regimes (Section 3); `F16` concept-drift
  alignment; and `F4` latent PCA / KMeans / autoencoder (fit per asset class).
- **Structure and context.** `F7` microstructure (volume and open interest); `F10` OHLC price action;
  `F8` calendar sin/cos; `F9` cross-sectional rank and pair-correlation (motivated by the blocks in §1.3
  and the low cross-signal correlation); and `F11` macro context (Section 2).

`F14` is intentionally skipped, a gap left when the three branches were merged. Per-column documentation
for every feature is in the `stml.metamodel.catalog` module.

### 4.3 — How the matrix is built

```python
from stml.io import load_clean_data
from stml.metamodel import FeaturePipeline
ohlcv, signals = load_clean_data()
matrix = FeaturePipeline().fit(ohlcv, signals).transform(ohlcv, signals)
```

`.fit()` fixes the chronological train/val/test split, fits the **TF** families on the FE-train block
only, and freezes every learned parameter. `.transform()` then applies the engineered **E** families and
the frozen **TF** transforms causally. It adds the cross-sectional and z-twin columns, **keeps only the
nonzero-signal trade-days**, tags each row's `partition` and `fe_train_end_date`, and preserves
structural NaNs (a closed venue is itself information, so it is never forward-filled). The CLI entry point
is `python -m stml.metamodel.build_features`, and the artifact it writes is
`results/feature_matrix.parquet`.

In [ ]:
# Provenance — the freshly built matrix, summarised live (not read from a committed artifact).
print("provenance (this run):")
for k in ["fe_train_end_date", "seed", "n_rows", "n_feature_cols"]:
    print(f"  {k}: {prov[k]}")
print(f"  partition_row_counts: {prov['partition_row_counts']}")
assert matrix.shape == (prov["n_rows"], prov["n_feature_cols"] + 4)
print(f"OK — matrix {matrix.shape} == {prov['n_rows']} rows x ({prov['n_feature_cols']} features + 4 metadata)")

## Section 5 — Reproducibility & leakage discipline (summary)

- **Regenerable from raw inputs.** Every feature in `results/feature_matrix.parquet` is rebuilt in
  Section 0 from `data/ohlcv_data.csv` and `data/primary_signals.csv` (plus the F11 macro workbook for
  the released window, or the shipped OOS CSV for the hidden window). No engineered artifact is
  hand-edited.
- **Fit on train, then freeze.** All TF families are fit on the FE-train block (≤ 2021-07-01) and frozen.
  The boundary is pinned by *date*, so extending the data to the hidden window cannot move it
  (`pipe._macro.train_index.max() == 2021-07-01`, and the same holds for each per-class F4 latent fit).
- **Causality.** The E families are causal by **truncation-invariance**, and **structural NaNs are
  preserved** (a closed venue is information, so it is never forward-filled).
- **Determinism.** The seed is 42, and the released-window rebuild reproduces the committed matrix to
  about 1e-10.
- **Course techniques cited.** Unsupervised structure and clustering (Session 2), HMM regimes
  (Session 3), and the supervised time-series feature pipeline (Session 4).
- **Relationship to Part II.** This matrix is Part I's **feature-engineering deliverable** (rubric
  Section 1): a leakage-disciplined, model-ready library that demonstrates the techniques. The graded
  meta-model in Part II is trained on the team's **integrated feature stack**, a separate parallel
  pipeline (`stml.harry` and `new_work`), and it does **not** read `results/feature_matrix.parquet`. The
  Part II intro explains this seam.

---
# Part II — Meta-model: learning, weight extraction, sizing & hyper-parameter cache

Part I built the feature set. Part II is the **graded meta-model**, and it does five things:

1. It fits a classifier that scores each non-zero primary signal with the probability that the trade is
   profitable.
2. It selects one **champion model per instrument** using **purged combinatorial cross-validation
   (CPCV)**.
3. It extracts the **cluster-level weight vector** (the feature importance) of each champion.
4. It calibrates the probabilities and uses them to size positions.
5. It saves the chosen model and hyper-parameters so a re-run reuses them instead of searching again.

This is the team's `new_work` meta-model pipeline (`stml.new_work` + `stml.harry` + the import-closed
`stml.experimental` strategy layer), run here in the notebook. It maps onto the rubric as follows. The
triple-barrier **labels** come from the team's canonical CSV with next-day entry (§6). Section 7
compares four model families, pooled and per-instrument, and locks a champion and feature variant per
instrument. Section 8 saves that selection. Section 9 reports cluster-level feature importance. Section
10 evaluates the model. Section 11 constructs the strategy and sizes positions.

> **Run model.** Part II **loads the team's committed results by default** (`FORCE_RECOMPUTE = False`),
> so it runs in seconds and reproduces the published numbers exactly. Set `FORCE_RECOMPUTE = True` to
> re-run the full CPCV search and fit (several hours) and to rebuild the §11 strategy returns from the
> probabilities. Methods A and B (including **bsops**) rebuild in about 30 seconds. The neural sizers C
> and D recompute only when their `.pt` checkpoints are present; otherwise they are shown from committed
> results.

> **Architectural note — a separate feature system.** Part II uses its own feature matrix and labels,
> built by `stml.new_work.feature_importance` from `stml.harry` features and the team triple-barrier
> labels, with the model train/test split fixed at **2021-10-06**. This is deliberately separate from
> Part I's F1–F17 `FeaturePipeline` matrix and its 2021-07-01 boundary, and it does **not** read
> `results/feature_matrix.parquet`. Running the two systems separately is what lets us reproduce the
> team's published model results exactly.

In [ ]:
# --- Part II setup: meta-model artifacts, recompute switch, capability probe ------------------
import json
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

# Load by default (fast; reproduces the committed results). True = re-run the real CPCV fit (slow).
FORCE_RECOMPUTE = False

from stml.new_work import hp_cache, split_config
NW_OUT  = hp_cache.OUTPUTS         # src/stml/new_work/outputs
MC_OUT  = hp_cache.MC              # .../model_comparison
IMP_OUT = hp_cache.IMPORTANCE      # .../importance
HP_PATH = hp_cache.CACHE_PATH      # .../selected_hps.json

# Importing the model modules confirms the merged stml.new_work + stml.harry surface resolves cleanly.
from stml.new_work import (
    model_comparison, equity_model_comparison, energy_model_comparison,
    metals_model_comparison, champion_importance,
)

print("Part II mode:", "RECOMPUTE (real CPCV fit)" if FORCE_RECOMPUTE else "LOAD (committed artifacts)")
print("train/test cut:", split_config.GLOBAL_CUT.date(), "| embargo end:", split_config.EMBARGO_END.date())
print("artifacts root:", NW_OUT.relative_to(ROOT))
print("saved HP set present:", HP_PATH.exists())

In [ ]:
# --- Optional: re-run the real CPCV pipeline from raw inputs (FORCE_RECOMPUTE only) -----------
# Dependency order: model zoo -> champion importance (writes cluster_membership) -> per-class variant
# comparison. Every stage reads the team feature matrix + labels via
# stml.new_work.feature_importance. This is MULTI-HOUR; the committed artifacts already hold these
# exact results, so the default LOAD path reproduces them without recomputing.
if FORCE_RECOMPUTE:
    print("Re-running the CPCV pipeline from raw inputs (this can take hours)...")
    model_comparison.run_all(force=True)                      # _cache/*.parquet + selection_table.csv
    champion_importance.run_champion_importance(force=True)   # importance/*/ (incl. cluster_membership)
    equity_model_comparison.run(force=True)                   # model_comparison/equity/*
    energy_model_comparison.run(force=True)                   # model_comparison/energy/*
    metals_model_comparison.run(force=True)                   # model_comparison/metals/*
    print("Recompute complete — artifacts regenerated under", NW_OUT.relative_to(ROOT))
else:
    print("LOAD mode — using the committed CPCV artifacts under", NW_OUT.relative_to(ROOT))

## Section 6 — Labels, sample weights & the CPCV protocol

The meta-model learns from **triple-barrier meta-labels** `label ∈ {0,1}`, where 1 means following the
primary signal was profitable. We use the team's canonical labels. Entry is on the **next day**: the
signal is read at the close of day *t*, and the return is measured over *t+1 … t+h*. Each event also
carries an **average-uniqueness** weight (AFML Ch. 4). Overlapping label windows share return periods,
and this weight stops those overlaps from being counted more than once.

Cross-validation is **combinatorial purged k-fold** (`CombinatorialPurgedKFold`). With 6 groups taken
2 at a time it produces 15 test paths, and it purges and embargoes the data around each test block to
remove leakage. A single train/test cut at **2021-10-06** (about the 70th percentile of pooled events)
holds out the final block for an out-of-sample read.

In [ ]:
# Event / label summary from the team's canonical triple-barrier labels.
labels = pd.read_csv(DATA / "meta" / "triple_barrier_labels.csv", parse_dates=["date", "t1"])
pos = labels["label"].mean()
print(f"{len(labels):,} labelled events across {labels['instrument'].nunique()} instruments | "
      f"overall profitable share: {pos:.1%}")
tr = split_config.apply_train_mask(labels); te = split_config.apply_test_mask(labels)
print(f"train (<= {split_config.GLOBAL_CUT.date()}): {len(tr):,} events | "
      f"test (> {split_config.EMBARGO_END.date()}, sealed): {len(te):,} events")
bal = (labels.groupby("instrument")["label"].agg(["size", "mean"])
       .rename(columns={"size": "n_events", "mean": "profitable_share"}).round(3))
bal.index = [TICKER_NAMES.get(i, i) for i in bal.index]
display(bal)

### 6.1 — Choosing the barrier geometry by grid search

The barrier widths are **not assumed**; we choose them **per instrument by grid search** (rubric
Section 2). A model-free study sweeps **343 geometries**: profit-take and stop-loss multiples
`pt, sl ∈ {0.25 … 2.5}` in all asymmetric pairs, and a holding horizon `h ∈ {1, 2, 3, 5, 10, 15, 20}`.
For each geometry it labels every primary-signal event `y ∈ {0,1}`, where 1 means the trade was
profitable under that profit-take, stop-loss and horizon. Each geometry is then ranked **per instrument**
by the Sharpe ratio of its adjusted-PnL curve, which trades only the events the label keeps
(`s·y·r_{t+1}`), measured over 2020 to 2022-01. We keep the top 3 per instrument and **validate them on
the 2022-H1 hold-out**. As a placebo-in-time check, the whole search is repeated at return lags 0, 1 and
2: a real edge requires the tradeable **lag-1** result to beat the contemporaneous **lag-0** result.

The figure below shows, per instrument, two cumulative-PnL curves under that instrument's chosen
geometry: **raw** (follow the primary signal on every event, in blue) and **adjusted** (trade only the
profitable-labelled events, in orange). The dotted line marks the train/hold-out split, and each panel
title gives the chosen `pt / sl / h`. The adjusted curve is the **best case a meta-model could reach**:
it is the equity an oracle would earn if it predicted the triple-barrier label perfectly every time. The
distance between the orange and blue curves is the extra return the meta-model (§7–§11) aims to capture,
and the fact that this gap survives into the hold-out is what supports the chosen geometries.

In [ ]:
# Theoretical ceiling — per-instrument raw vs oracle-adjusted equity under each instrument's
# grid-searched best triple-barrier geometry (lag-1; dotted line = train/hold-out split).
# Source: the model-free barrier-selection study (triple-barrier-label grid search).
_fig = ROOT / "results" / "triple_barrier_oracle_pnl.png"
if _fig.exists():
    display(Image(filename=str(_fig)))
else:
    print("theoretical-ceiling figure not found:", _fig)


## Section 7 — Model comparison & champion selection

We select the meta-model per instrument with **combinatorial purged CV** (CPCV, `n_groups=6, k=2,
embargo=0.01`, giving 15 test paths), fitted on the **train block only** (events on or before
2021-10-06). For each instrument we score **four model families**: an elastic-net **logistic**
regression, a **random forest**, **XGBoost**, and an **MLP**. Each family is tuned per fold with an
inner purged k-fold and the 1-SE rule. We fit every family two ways: **per-instrument**, and on
**pooled** groups (`energy_cl_ho`, `energy_all`, `precious`). Pooling helps the thin single-name slices.
For example, `ho1s` and `ng1s` have few events of their own, so their champions are pooled-energy MLPs
scored on each name's own CPCV slice. The **champion** for an instrument is the (group, model) pair with
the highest **lower bound of the AUC confidence interval**. The `signal = True` flag marks instruments
whose champion clears 0.5 AUC with margin.

Each champion is then compared across **feature variants**: **full**, **pruned** and **reduced** (and,
where useful, `reduced_min` and `pca_reduced`). We pick the **locked** variant with a **1-SE rule**:
take the simplest variant whose CPCV mean AUC is within one cross-path standard deviation of the full
model (`auc_mean ≥ full_mean − full_std`). If no smaller variant qualifies, we keep the full model.

We run this in two phases to avoid selection bias. **Phase 1** does all selection on the train CPCV and
commits the locked variant **before** the test set is read. **Phase 2** then scores that locked variant
**once** on the held-out test set (events after the 2021-10-20 embargo). The non-locked variants are
reported as diagnostics only, and are never used to re-select. The difference between the Phase-1 CPCV
AUC and the Phase-2 out-of-sample AUC is our measure of how well the model generalises.

In [ ]:
# Step-3 champion: best lower-CI CPCV AUC across the 4-family zoo x {per-instrument, pooled groups}.
sel = pd.read_csv(MC_OUT / "selection_table.csv")
show = sel[["instrument", "best_group", "best_model", "best_auc", "lower_ci", "signal", "runner_up_model"]].copy()
show.insert(1, "name", [TICKER_NAMES.get(i, i) for i in show["instrument"]])
print("Champion per instrument (family x {per-instrument, pooled}, by lower-CI CPCV AUC):")
display(show.round(4))
sig = sel.loc[sel["signal"], "instrument"].tolist()
print(f"Tradeable meta-signal (champion clears 0.5 with margin): {len(sig)}/{len(sel)} -> {', '.join(sig)}\n")

# Phase 1 - CPCV AUC by feature variant. LOCK = simplest variant with auc_mean >= full_mean - full_std.
for cls in ["equity", "energy", "metals"]:
    cp = pd.read_csv(MC_OUT / cls / "cpcv_results.csv")
    locked = pd.read_csv(MC_OUT / cls / "locked_picks.csv").set_index("inst")["locked_variant"].to_dict()
    thr = {}
    for inst, g in cp.groupby("inst"):
        f = g[g["variant"] == "full"]
        if len(f):
            thr[inst] = float(f["auc_mean"].iloc[0] - f["auc_std"].iloc[0])
    cp["lock_thresh"] = cp["inst"].map(thr)
    cp["LOCKED"] = ["<<" if locked.get(i) == v else "" for i, v in zip(cp["inst"], cp["variant"])]
    print(f"{cls.upper()} - Phase-1 CPCV AUC by variant (lock_thresh = full_mean - full_std):")
    display(cp[["inst", "variant", "auc_mean", "auc_std", "logloss", "brier", "n_paths", "lock_thresh", "LOCKED"]].round(4))
    for inst in cp["inst"].unique():
        p = MC_OUT / cls / f"{inst}_cpcv_chart.png"
        if p.exists():
            display(Image(filename=str(p)))


In [ ]:
# Phase 2 - single-shot OOS on the sealed test (date > 2021-10-20 embargo). Locked variant = HEADLINE.
gap_rows = []
for cls in ["equity", "energy", "metals"]:
    oos = pd.read_csv(MC_OUT / cls / "oos_results.csv")
    cp = pd.read_csv(MC_OUT / cls / "cpcv_results.csv")
    o = oos.copy()
    o["role"] = ["HEADLINE" if x else "diagnostic" for x in o["is_locked"]]
    o["oos_95ci"] = [f"[{lo:.2f}, {hi:.2f}]" for lo, hi in zip(o["auc_ci_lo"], o["auc_ci_hi"])]
    print(f"{cls.upper()} - Phase-2 OOS AUC (single-shot; non-locked rows are diagnostics only):")
    display(o[["inst", "variant", "role", "auc", "oos_95ci", "logloss", "brier", "n_test"]].round(4))
    p = MC_OUT / cls / "oos_summary_chart.png"
    if p.exists():
        display(Image(filename=str(p)))
    for _, r in oos[oos["is_locked"]].iterrows():
        dev = cp[(cp["inst"] == r["inst"]) & (cp["variant"] == r["variant"])]
        d = float(dev["auc_mean"].iloc[0]) if len(dev) else float("nan")
        gap_rows.append({
            "instrument": r["inst"], "name": TICKER_NAMES.get(r["inst"], r["inst"]),
            "locked": r["variant"], "dev_cpcv_auc": round(d, 4), "oos_auc": round(r["auc"], 4),
            "oos_95ci": f"[{r['auc_ci_lo']:.2f}, {r['auc_ci_hi']:.2f}]", "n_test": int(r["n_test"]),
            "dev_minus_oos": round(d - r["auc"], 4),
        })
gap = pd.DataFrame(gap_rows)
print("Dev-CPCV -> OOS generalisation gap (locked variant only; positive = train-CV optimism):")
display(gap)
print("Reads: rb1s transfers cleanly (gap ~0.00, OOS AUC 0.72, CI fully above 0.5); ng1s OOS exceeds "
      "its dev CPCV (negative gap). Thin sealed slices (ho1s n=27, gc1s n=23) carry very wide CIs; "
      "hg1s / pl1s show the largest optimism. Most single-name OOS CIs still straddle 0.5 - the meta-"
      "signal is real but modest, as the EDA anticipated.")


## Section 8 — Saved hyper-parameter / champion set

The CPCV search is expensive, and its result is a per-instrument selection: which model family, which
feature variant, and which feature list. We consolidate that selection into
**`outputs/selected_hps.json`**. The first run writes the file; every later run loads it and skips the
search. The low-level estimator settings (such as tree depth or learning rate) are re-tuned per fold by
the 1-SE rule, so the file stores the **selection**, which is the costly part of the pipeline, rather
than a single fixed set of estimator parameters.

In [ ]:
if FORCE_RECOMPUTE or not HP_PATH.exists():
    hp = hp_cache.build_selected_hps(write=True)
    print(f"Saved hyper-parameter set -> {HP_PATH.relative_to(ROOT)}  ({hp['meta']['n_instruments']} instruments)")
else:
    hp = hp_cache.load_selected_hps()
    print(f"Loaded saved hyper-parameter set <- {HP_PATH.relative_to(ROOT)}  ({hp['meta']['n_instruments']} instruments)")

rows = []
for inst, r in hp["instruments"].items():
    rows.append({"instrument": inst, "name": TICKER_NAMES.get(inst, inst),
                 "class": r.get("asset_class"), "champion": r.get("champion_model"),
                 "locked_variant": r.get("locked_variant"),
                 "signal": r.get("signal"), "weight_vector": r.get("weight_vector_file")})
display(pd.DataFrame(rows).set_index("instrument"))

## Section 9 — Weight-vector extraction (cluster-level feature importance)

The **weight vector of the meta-model** is the per-feature importance of each instrument's champion. We
compute it with a workflow that uses train-only data inside CPCV and accounts for collinearity:

1. **Cluster the features.** We turn Spearman correlations into a distance `√(1 − |ρ|)` over the
   continuous features, link them with **Ward** clustering, and choose the number of clusters K by the
   **silhouette** score (we also report Calinski-Harabasz and Davies-Bouldin). The signal-derived (F5),
   latent (F4) and calendar (F8) families are assigned to groups by hand.
2. **Score each cluster.** Inside the purged CPCV loop we compute **clustered MDA**, the drop in AUC when
   a whole cluster's features are permuted together, as the main model-agnostic measure. For tree
   champions we add **clustered MDI** and **group tree-SHAP**; for logistic champions we add the **summed
   standardised |coef|**. A cluster counts as significant when its mean AUC drop exceeds one standard
   deviation of its path-to-path noise.
3. **Check agreement across methods.** We compute **Kendall-τ** between the rankings (MDA vs MDI vs SHAP,
   or MDA vs coefficient). MDI and SHAP usually agree closely (τ ≈ 0.9), while MDA is permutation-based
   and is the strictest test.
4. **Look inside the top clusters.** For each top cluster we rank its members by |SHAP| or |coef| and fit
   a **PCA**. If the first principal component explains most of the variance, the cluster behaves as a
   single latent factor, and we represent it by that top component (this is how the logistic
   `pca_reduced` sets are built). If not, the cluster is genuinely multi-dimensional.
5. **Report per feature.** Finally we save, per instrument, the per-feature weight vector: the signed
   **standardised coefficients** for logistic champions, or the **MDI and tree-SHAP magnitudes** for tree
   champions.

In [ ]:
# (a) The weight vector: top-N importance per champion (signed; the meta-model's learned directions).
TOP = 8
print(f"Weight vector - top-{TOP} features per champion (sign = direction of P(profit)):\n")
for inst, r in hp["instruments"].items():
    wv = r.get("weight_vector_file")
    p = IMP_OUT / inst / (wv or "")
    if not wv or not p.exists():
        continue
    df = pd.read_csv(p)
    magcol = "coef_abs" if "coef_abs" in df.columns else "shap_magnitude"
    sgncol = "coef_signed" if "coef_signed" in df.columns else "shap_signed"
    top = df.sort_values(magcol, ascending=False).head(TOP)
    kind = "|coef|" if magcol == "coef_abs" else "|SHAP|"
    feats = ", ".join(f"{f} {'+' if s >= 0 else '-'}{m:.3f}"
                      for f, m, s in zip(top["feature"], top[magcol], top[sgncol]))
    print(f"{TICKER_NAMES.get(inst, inst):22s} [{r.get('champion_model'):8s} {kind}]  {feats}")

# (b) Cross-instrument cluster-importance summary: family, signal, #significant clusters, top cluster, agreement.
rows = []
for inst in sorted(d.name for d in IMP_OUT.iterdir() if d.is_dir()):
    m = pd.read_csv(IMP_OUT / inst / "champion_meta.csv").iloc[0].to_dict()
    mda = pd.read_csv(IMP_OUT / inst / "clustered_mda_full.csv").sort_values("mean_drop", ascending=False)
    ra = pd.read_csv(IMP_OUT / inst / "rank_agreement.csv")
    tau = ", ".join(f"{x.method_pair}={x.kendall_tau:+.2f}" for x in ra.itertuples())
    rows.append({"instrument": inst, "name": TICKER_NAMES.get(inst, inst), "family": m["family"],
                 "model": m["model_type"], "auc": f"{m['auc_mean']:.3f}±{m['auc_std']:.3f}",
                 "lower_ci": round(m["lower_ci"], 3), "signal": m["signal"],
                 "n_sig_clusters": m["n_sig_clusters"], "top_cluster": mda.iloc[0]["cluster"],
                 "top_mda": round(mda.iloc[0]["mean_drop"], 4), "rank_agreement_tau": tau})
print("\nCross-instrument cluster-importance summary (one champion per instrument):")
display(pd.DataFrame(rows))

# (c) Cluster-level cross-check: MDA vs MDI vs SHAP for a tree champion; MDA vs Coef for a logistic one.
for rep in ["nq1s", "es1s"]:
    cc = pd.read_csv(IMP_OUT / rep / "cluster_crosscheck_table.csv")
    print(f"\n{TICKER_NAMES.get(rep, rep)} ({rep}) - cluster-level importance cross-check (top 6 by MDA):")
    display(cc.head(6).round(4))

# (d) Within-cluster breakdown for the top cluster of a logistic champion; PCA exposes collinearity.
rep = "es1s"
mda = pd.read_csv(IMP_OUT / rep / "clustered_mda_full.csv").sort_values("mean_drop", ascending=False)
topc = mda.iloc[0]["cluster"]
wcp = IMP_OUT / rep / f"within_cluster_{topc}.csv"
if wcp.exists():
    wc = pd.read_csv(wcp)
    magcol = next(c for c in ("mean_shap_mag", "mean_coef_abs", "score") if c in wc.columns)
    pc1 = wc["pca_pc1_var_explained"].iloc[0] if "pca_pc1_var_explained" in wc.columns else float("nan")
    print(f"\n{TICKER_NAMES.get(rep, rep)} - within-cluster ranking for {topc} "
          f"(PC1 explains {pc1:.0%} of cluster variance):")
    display(wc[["feature", magcol, "pc1_loading"]].sort_values(magcol, ascending=False).round(4))
    wimg = IMP_OUT / rep / f"within_cluster_{topc}.png"
    if wimg.exists():
        display(Image(filename=str(wimg)))

# (e) Representative charts per asset class: Spearman/Ward dendrogram, cluster MDA, global weight vector.
for rep in ["es1s", "nq1s", "cl1s", "pl1s"]:
    wv = hp["instruments"].get(rep, {}).get("weight_vector_file", "")
    gchart = "global_coef_chart.png" if "coef" in wv else "global_shap_chart.png"
    shown = [img for img in ["dendrogram.png", "clustered_mda_chart.png", gchart]
             if (IMP_OUT / rep / img).exists()]
    if shown:
        print(f"\n=== {TICKER_NAMES.get(rep, rep)} ({rep}) - dendrogram | cluster MDA | global weight vector ===")
        for img in shown:
            display(Image(filename=str(IMP_OUT / rep / img)))


## Section 10 — Model evaluation (classification metrics & baseline)

We evaluate on the held-out out-of-sample window (events after the 2021-10-20 embargo), using the
calibrated champion probabilities in `outputs/metamodel_predictions.csv`. We report the classification
metrics the rubric asks for — **precision, recall, F1 and AUC** — together with a **confusion matrix**
and a **decision-threshold sweep**. We report these both pooled and **per instrument**. The baseline for
comparison is the strategy that takes every primary-signal event without filtering.

The rubric marks methodology rather than performance, so here is the plain reading. The meta-model is a
**selective filter**. At a threshold of `p > 0.5` it trades about 54% of events, at higher precision than
the take-all base rate. The take-all baseline still wins on raw **F1**, because taking every event forces
recall to 1 and inflates F1. The meta-model's value shows up instead in **precision, in ranking (AUC),
and in the risk-adjusted strategy** of §11. The effect also **varies by instrument**. On some
instruments the model ranks events well on AUC but, after calibration, their low base rate maps below the
0.5 threshold, so the model abstains and takes nothing. It helps modestly on several equity and energy
names, and it adds nothing on others (AUC ≈ 0.5). We report this per instrument rather than hide it
behind a single pooled number.

In [ ]:
# Re-assert the inline backend (the Part II chart modules set Agg at import), then evaluate.
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (roc_auc_score, precision_score, recall_score, f1_score,
                             accuracy_score, confusion_matrix)

_mp = pd.read_csv(NW_OUT / "metamodel_predictions.csv", parse_dates=["date"])
y = _mp["bin"].astype(int).to_numpy()
p = _mp["calibrated_proba"].to_numpy()
base = y.mean()
pred = (p > 0.5).astype(int)

# (a) Pooled metrics @0.5 vs the blind-primary baseline (take every event).
summary = pd.DataFrame([
    {"model": "meta-model (p>0.5)", "events_taken": f"{pred.mean():.0%}",
     "precision": round(precision_score(y, pred), 3), "recall": round(recall_score(y, pred), 3),
     "F1": round(f1_score(y, pred), 3), "accuracy": round(accuracy_score(y, pred), 3),
     "AUC": round(roc_auc_score(y, p), 3)},
    {"model": "baseline: follow primary (take all)", "events_taken": "100%",
     "precision": round(base, 3), "recall": 1.0, "F1": round(2 * base / (base + 1), 3),
     "accuracy": round(base, 3), "AUC": 0.5},
])
print(f"OOS events = {len(y)} | profitable base rate = {base:.1%}")
display(summary)

# (b) confusion matrix @0.5 | (c) threshold sweep | (d) per-instrument AUC
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
cm = confusion_matrix(y, pred)
ax[0].imshow(cm, cmap="Blues")
ax[0].set_xticks([0, 1]); ax[0].set_xticklabels(["pred 0", "pred 1"])
ax[0].set_yticks([0, 1]); ax[0].set_yticklabels(["true 0", "true 1"])
ax[0].set_title("Confusion matrix (p>0.5)")
for (r, c), v in np.ndenumerate(cm):
    ax[0].text(c, r, int(v), ha="center", va="center",
               color="white" if v > cm.max() / 2 else "black", fontsize=12)
taus = np.linspace(0.30, 0.65, 36)
ax[1].plot(taus, [precision_score(y, (p > t).astype(int), zero_division=0) for t in taus], label="precision")
ax[1].plot(taus, [recall_score(y, (p > t).astype(int), zero_division=0) for t in taus], label="recall")
ax[1].plot(taus, [f1_score(y, (p > t).astype(int), zero_division=0) for t in taus], label="F1")
ax[1].axvline(0.5, color="0.7", ls="--", lw=0.8)
ax[1].axhline(base, color="crimson", ls=":", lw=1, label=f"base rate {base:.2f}")
ax[1].set_xlabel("decision threshold"); ax[1].set_title("Threshold sweep"); ax[1].legend(fontsize=8)
per = []
for inst, g in _mp.groupby("instrument"):
    try:
        a = roc_auc_score(g["bin"].astype(int), g["calibrated_proba"])
    except ValueError:
        a = np.nan
    per.append((TICKER_NAMES.get(inst, inst), a))
per.sort(key=lambda x: np.nan_to_num(x[1]))
ax[2].barh([x[0] for x in per], [x[1] for x in per],
           color=["#4c72b0" if (a or 0) >= 0.5 else "#c44e52" for _, a in per])
ax[2].axvline(0.5, color="k", lw=0.8); ax[2].set_xlim(0.3, 0.8)
ax[2].set_title("Per-instrument OOS AUC"); ax[2].tick_params(labelsize=8)
plt.tight_layout(); plt.show()

# (e) Per-instrument table (AUC + precision/recall/F1 @0.5).
prow = []
for inst, g in _mp.groupby("instrument"):
    yy = g["bin"].astype(int).to_numpy(); pp = g["calibrated_proba"].to_numpy()
    pr = (pp > 0.5).astype(int)
    try:
        a = round(roc_auc_score(yy, pp), 3)
    except ValueError:
        a = float("nan")
    prow.append({"instrument": inst, "name": TICKER_NAMES.get(inst, inst), "n": len(g),
                 "base": round(yy.mean(), 3), "AUC": a, "taken": f"{pr.mean():.0%}",
                 "precision": round(precision_score(yy, pr, zero_division=0), 3),
                 "recall": round(recall_score(yy, pr, zero_division=0), 3),
                 "F1": round(f1_score(yy, pr, zero_division=0), 3)})
print("Per-instrument OOS classification (calibrated p>0.5), sorted by AUC:")
display(pd.DataFrame(prow).sort_values("AUC", ascending=False))
print("Note rb1s/si1s: strong AUC ranking but 0% taken at p>0.5 — calibration maps their low base rates "
      "below the gate, so the meta-model abstains rather than trade a weak edge (consistent with §11).")


## Section 11 — Strategy construction & position sizing

For each non-zero primary signal, the champion meta-model produces a probability, **P(trade
profitable)**. This section uses those probabilities to size positions and build a portfolio, and it
answers the graded question: does the meta-model improve on simply following the primary signal?

**From scores to calibrated probabilities.** A raw model score is not yet a reliable probability, so we
apply **Platt scaling**: a logistic fit on the out-of-fold scores. After calibration, a predicted 0.6
corresponds to roughly a 60% realised hit rate. The calibrated out-of-sample probabilities are in
`outputs/metamodel_predictions.csv` (columns `raw_proba` and `calibrated_proba`), and the training
out-of-fold set is `data/oof_meta_probabilities.csv`.

**Portfolio mechanics.** We estimate each instrument's daily volatility with an **EWMA (span = 60)** of
close-to-close returns, annualised by √252, with a 2% floor. Each position is then scaled to a target
volatility of σ_tgt = 10% and clipped to ±10×. The portfolio gives the 11 instruments equal risk weight,
holds cash when flat, and lags positions by one day, so a weight set at the close of day *t* earns the
return from *t* to *t+1*. Returns are net of Grinold-Kahn transaction costs: a 2 bps half-spread plus
10 bps per unit of weight traded (10 bps × |Δw|).

**Sizing methods.**
- **A — primary (benchmark).** Follow the signal at full conviction (ŷ in {-1, 0, 1}). No meta-model.
- **B — meta-filtered.** Scale conviction by the calibrated probability. The sizing methods are
  `all_or_nothing` (a gate at p̂ > 0.5), `model_confidence`, `ncdf`, and **SOPS**, a sigmoid
  `f(p) = 1/(1+e^{-(a·p − c)})` whose `(a, c)` are fit on the out-of-fold set to **maximise training
  Sharpe**. The selection we committed to in advance is **bsops = Method B + SOPS**
  (`stml.experimental.sizing`).
- **C and D — neural sizers.** A **VSN-LSTM** and a **Temporal Fusion Transformer**
  (`stml.new_work.models`) learn conviction directly from sequence features. We show them from saved
  checkpoint inference. Their `.pt` weights are not shipped, so the recompute path skips C and D unless
  the weights are placed in `outputs/`. Methods A and B, including bsops, always recompute.

The out-of-sample window is **2021-10-21 to 2022-06-29** (the held-out test, with BOUNDARY = 2021-10-06).
This strategy layer is the team's `stml.new_work.evaluate` and `stml.experimental` pipeline, and it is
separate from Part I's primary-signal portfolio cell.

In [ ]:
# Re-assert the inline backend (the Part II chart modules set Agg at import).
%matplotlib inline
# --- Section 11 setup: strategy artifacts + recompute-capability probe -----------------------
import numpy as np
import matplotlib.pyplot as plt

SE_DIR = ROOT / "results" / "strategy_eval"      # committed daily net returns + summary + charts
MP_PATH = NW_OUT / "metamodel_predictions.csv"   # OOS events + raw/calibrated probabilities
OOF_PATH = DATA / "oof_meta_probabilities.csv"   # training out-of-fold probabilities (SOPS fit)

from stml.new_work import evaluate as strat_eval  # import-closed strategy runner (methods A-D)

have_vsn = (NW_OUT / "vsn_lstm_weights_cp5.pt").exists() or (NW_OUT / "vsn_lstm_weights.pt").exists()
have_tft = (NW_OUT / "tft_weights_cp5.pt").exists() or (NW_OUT / "tft_weights.pt").exists()

print("Strategy mode:", "RECOMPUTE (rebuild net returns)" if FORCE_RECOMPUTE else "LOAD (committed net returns)")
print("OOS predictions present:", MP_PATH.exists(), "| OOF probs present:", OOF_PATH.exists())
print(f"Neural checkpoints -> C-VSN-LSTM: {have_vsn} | D-TFT: {have_tft}"
      + ("" if (have_vsn and have_tft) else "  (absent -> C/D shown from committed net returns)"))


In [ ]:
# --- Optional: rebuild the strategy net returns from the probabilities (FORCE_RECOMPUTE only) -
# evaluate.run() always rebuilds A + B-aon, adds B-sops (bsops) when the OOF probs exist, and adds
# C/D only when their .pt checkpoints are present (else it prints a skip line). Deterministic
# (seed=42); regenerates results/strategy_eval/net_returns_*.csv identical to the committed set.
if FORCE_RECOMPUTE:
    strat_eval.run(include_sops=True, include_neural=True, verbose=False)
    print("Strategy net returns regenerated under", SE_DIR.relative_to(ROOT))
else:
    print("LOAD mode - using committed net returns under", SE_DIR.relative_to(ROOT))


In [ ]:
# (a) Headline: metamodel-enhanced (Method B) vs primary (Method A), from the committed summary.
summary = pd.read_csv(SE_DIR / "eval_summary.csv", index_col=0)
print("Primary (A) vs meta-filtered (B all-or-nothing / B SOPS) - OOS 2021-10-21 -> 2022-06-29:")
display(summary)

# (b) Sizing zoo - annualised Sharpe / vol / maxDD, live from each method's daily net returns.
ZOO = {"A": "A - primary", "B_aon": "B - all-or-nothing", "B_mc": "B - model-confidence",
       "B_ncdf": "B - ncdf", "B_sops": "B - SOPS (bsops, selected)",
       "C_vsn_lstm": "C - VSN-LSTM", "D_tft": "D - TFT"}
zoo, curves = [], {}
for key, lab in ZOO.items():
    p = SE_DIR / f"net_returns_{key}.csv"
    if not p.exists():
        continue
    r = pd.read_csv(p, index_col=0, parse_dates=True).iloc[:, 0]
    cum = (1 + r).cumprod()
    curves[lab] = cum - 1.0
    dd = (cum / cum.cummax() - 1.0).min()
    zoo.append({"method": lab, "sharpe": round(r.mean() / r.std() * np.sqrt(252), 3),
                "ann_vol": f"{r.std() * np.sqrt(252):.2%}", "max_dd": f"{dd:.2%}", "n_days": len(r)})
print("\nSizing-method comparison (annualised, from daily net returns):")
display(pd.DataFrame(zoo))

# (c) Cumulative net return - primary vs the selected meta-filtered sizers (live replot).
fig, ax = plt.subplots(figsize=(9, 4.2))
for lab in ["A - primary", "B - all-or-nothing", "B - SOPS (bsops, selected)"]:
    if lab in curves:
        ax.plot(curves[lab].index, curves[lab].values, lw=1.7, label=lab)
ax.axhline(0, color="0.7", lw=0.8)
ax.set_ylabel("cumulative net return"); ax.legend(loc="upper left")
ax.set_title("OOS cumulative net return - primary vs meta-filtered (vol-targeted, net of costs)")
plt.tight_layout(); plt.show()

# (d) Calibration: probability histogram + reliability curve (raw vs Platt-calibrated vs realised).
mp = pd.read_csv(MP_PATH, parse_dates=["date"])
fig, (axh, axr) = plt.subplots(1, 2, figsize=(11, 4))
axh.hist(mp["calibrated_proba"], bins=30, color="#4c72b0", alpha=0.85)
axh.axvline(0.5, color="crimson", ls="--", lw=1)
axh.set_title("Calibrated P(profit) - OOS"); axh.set_xlabel("calibrated probability"); axh.set_ylabel("events")
for col, lab, c in [("raw_proba", "raw", "#999999"), ("calibrated_proba", "calibrated", "#4c72b0")]:
    q = pd.qcut(mp[col], 10, duplicates="drop")
    g = mp.groupby(q, observed=True).agg(pred=(col, "mean"), real=("bin", "mean"))
    axr.plot(g["pred"], g["real"], "o-", color=c, lw=1.4, label=lab)
axr.plot([0, 1], [0, 1], "k:", lw=1); axr.set_xlim(0.1, 0.8); axr.set_ylim(0.1, 0.8)
axr.set_xlabel("predicted P(profit)"); axr.set_ylabel("realised profitable share")
axr.set_title("Reliability (Platt-calibrated)"); axr.legend(loc="upper left")
plt.tight_layout(); plt.show()

taken = mp[mp["calibrated_proba"] > 0.5]; skipped = mp[mp["calibrated_proba"] <= 0.5]
print(f"Meta-filter at p>0.5 takes {len(taken)}/{len(mp)} events ({len(taken) / len(mp):.0%}); "
      f"taken realise {taken['bin'].mean():.1%} profitable vs skipped {skipped['bin'].mean():.1%}.")

# (e) Committed per-asset detail (dense grids shown as the shipped figures).
for img in ["model_comparison_cpcv.png", "per_asset_cumulative_returns.png", "per_instrument_contribution.png"]:
    ip = SE_DIR / img
    if ip.exists():
        display(Image(filename=str(ip)))

print("\nTakeaway: the meta-filter lifts OOS Sharpe 1.41 -> 1.61 (+14%) while halving vol, drawdown and "
      "cost (turnover 48 -> 22/yr), trading ~54% of events. bsops is the pre-committed selection; the "
      "model-confidence / ncdf variants edge it on this single OOS draw, while the neural sizers (C/D) "
      "did not transfer. t-stats ~1.2-1.4 over 180 days are not significant at 5% (PSR ~0.86-0.90).")


## Section 12 — Part II summary & deferred work

- **Labeling (§6, §6.1).** The model learns from the team's canonical triple-barrier meta-labels, with
  next-day (t+1) entry and average-uniqueness weighting. The barrier geometry is chosen per instrument by
  grid search, and the gap between the raw and oracle equity curves is what motivates building a
  meta-model at all.
- **Model comparison and variant lock (§7).** We compare four families (logistic, random forest,
  XGBoost, MLP), both per-instrument and pooled, scored over 15 CPCV paths. The champion is the one with
  the highest lower-bound AUC. We then compare full, pruned and reduced feature variants and lock the
  simplest variant within one standard deviation of the full model. The locked variant is confirmed once
  on the out-of-sample set, and we report the gap between cross-validated and out-of-sample AUC.
- **Saved hyper-parameters (§8).** The full selection is saved to `outputs/selected_hps.json` and loaded
  on every re-run.
- **Feature importance (§9).** We cluster features by Spearman distance, score each cluster (MDA, MDI,
  SHAP, or coefficients), look inside the top clusters with PCA, and check agreement across methods with
  Kendall-τ.
- **Model evaluation (§10).** We report precision, recall, F1 and AUC, a confusion matrix, a threshold
  sweep, and a per-instrument breakdown, all against the take-all primary baseline. The meta-model acts as
  a selective precision filter, and its effect varies by instrument.
- **Strategy and sizing (§11).** Using Platt-calibrated probabilities and several sizing methods
  (`all_or_nothing`, `model_confidence`, `ncdf`, **SOPS / bsops**, plus the VSN-LSTM and TFT sizers), the
  meta-filter raises the out-of-sample Sharpe from **1.41 to 1.61**, at roughly half the volatility,
  drawdown and cost.

**Deferred work.** Three items remain for a later step: the consolidated `date,instrument,prediction`
deliverable CSV; the refit on all released data that then predicts the hidden **Jul–Dec 2022** window
(the boundary moves via `STML_BOUNDARY`); and shipping the **C/D neural checkpoints** (`.pt`) so the
VSN-LSTM and TFT sizers can recompute live.